In [ ]:
!uv pip install pystac-client odc-stac folium dask distributed

In [ ]:
from pystac.client import Client
from odc.stac import load
from datacube_compute import geomedian_with_mads
from odc.geo import BoundingBox
from dask.distributed import Client as DaskClient

In [ ]:
# bbox over Hobart
bbox = BoundingBox(left=147.2, bottom=-42.9, right=147.4, top=-42.8)
bbox.explore()

In [ ]:
url = "https://earth-search.aws.element84.com/v1"
catalog = Client.open(url)

collection = "sentinel-2-l2a"
items = catalog.search(
    collections=[collection],
    bbox=bbox,
    datetime="2026-01/2026-03"
).item_collection()

print(f"Found {len(items)} items")

In [ ]:
data = load(items, bbox=bbox, measurements=["red", "green", "blue", "scl"], chunks={})

# Mask clouds, 3 is cloud shadow, 8 is clouds, 9 is thin cirrus.
clouds = data.scl.isin([3, 8, 9])
data = data.where(~clouds, 0)
data = data.drop_vars("scl")

# Do the calculation using Dask
with DaskClient() as dask_client:
    geomad = geomedian_with_mads(
        data,
        scale=10000,
        offset=0,
        nodata=0,
    ).compute()

geomad

In [ ]:
geomad.odc.explore(vmin=0, vmax=3000)